# Denison CS181/DA210 SW Lab #8 - Step 1

Before you get your checkpoints, make sure everything runs as expected. This is a combination of **restarting the kernel** and then **running all cells**.

Make sure you fill in any place that says `# YOUR CODE HERE` or "YOUR ANSWER HERE".

---

> **In the questions that follow, we are looking for XPath declarative solutions to the problems, not procedural solutions.  You will not get credit for procedural solutions.**

---

## Part A: XPath basics

As we've seen in class, XPath provides a powerful declarative alternative to XML procedural operations.

We'll summarize some basic XPath operations here, as well as some we haven't seen yet.

We'll use the `indicators` dataset in `ind0.xml`.

In [2]:
from lxml import etree
import os.path

datadir = "publicdata"

ind_path = os.path.join(datadir, "ind0.xml")
parser = etree.XMLParser(remove_blank_text=True)
tree = etree.parse(ind_path, parser)

ind_root = tree.getroot()

#### First child of the root

We can use the procedural operation `getchildren` and index into the resulting list:

In [3]:
first_child = ind_root.getchildren()[0]
print("First child:", first_child, "with tag:", first_child.tag) # how else?

First child: <Element country at 0x1e770fa4980> with tag: country


Alternatively, we can use XPath to match all nodes with the given tag:

In [4]:
first_child = ind_root.xpath("/indicators/country")[0]
print("First child:", first_child, "with tag:", first_child.tag)

First child: <Element country at 0x1e770fa4980> with tag: country


Or, we could use the built-in function `position()` to specify that we want the first node that matches the path.  (Note that `position()` indexes from 1, and not 0.)

In [5]:
first_child = ind_root.xpath("/indicators/country[position() = 1]")[0] # only one element
print("First child:", first_child, "with tag:", first_child.tag)

First child: <Element country at 0x1e770fa4980> with tag: country


#### Value of attribute

In procedural XML operations, we need to use the `attrib` dictionary or `.get()` to get the value of an attribute:

In [6]:
country_names = []
for country_node in ind_root:
    country_names.append(country_node.get("name"))
print(country_names)

['France', 'United Kingdom', 'United States']


In XPath, we can take another "step" in our path for the given attribute (but have no need to specify a loop):

In [7]:
country_names = ind_root.xpath("/indicators/country/@name")
print(country_names)

['France', 'United Kingdom', 'United States']


#### Children (tags) of a node

Similarly, in procedural XML, we need to use a loop to get the tags of all children of a node.

In [8]:
# Goal: tags below timedata for France in 2007
code = "FRA"
year = "2007"
tags = []
for country_node in ind_root:
    if country_node.get("code") != code: continue
    for timedata_node in country_node:
        if timedata_node.get("year") != year: continue
        for ind_node in timedata_node:
            tags.append(ind_node.tag)
print(tags)

['pop', 'gdp']


In XPath, this is significantly simpler, as we can filter on attributes:

In [9]:
code = "FRA"
year = "2007"
# Note: we have to be careful and escape the value of the attribute
# with '', so we use a string format to perform the match
path = f"/indicators/country[@code='{code}']/timedata[@year='{year}']/*"
ind_nodes = ind_root.xpath(path)
tags = [node.tag for node in ind_nodes]
print(tags)

['pop', 'gdp']


#### Text of a node

Using procedural XML operations, we could find all nodes with a given tag and get their text, but this again requires a loop:

In [10]:
pop_list = []
for pop_node in ind_root.iter("pop"):
    pop_list.append(pop_node.text)
print(pop_list)

['64.02', '66.87', '61.32', '66.06', '301.23', '325.15']


In XPath, we need only get the `text()` of each node in the path:

In [11]:
pop_list = ind_root.xpath("/indicators/country/timedata/pop/text()")
print(pop_list)

['64.02', '66.87', '61.32', '66.06', '301.23', '325.15']


Additionally, we can use a shortcut if we want all nodes with a given tag in the tree:

In [12]:
pop_list = ind_root.xpath("//pop/text()")
print(pop_list)

['64.02', '66.87', '61.32', '66.06', '301.23', '325.15']


#### Filtering on attributes and text

We can use XML procedural operations to get the countries with 2017 population less than 100 million:

In [13]:
small_list = []
for country_node in ind_root:
    for timedata_node in country_node:
        if timedata_node.get("year") != "2017": continue
        pop_node = timedata_node.find("pop")
        if float(pop_node.text) < 100:
            small_list.append(country_node.get("name"))
            break
print(small_list)

['France', 'United Kingdom']


With XPath, we can do this filtering within our path, with some extra steps at the end to backtrack up the tree to get the country's name:

In [14]:
small_list = ind_root.xpath("//timedata[@year='2017']/pop[text()<100]/../../@name")
print(small_list)

['France', 'United Kingdom']


---

## Part B: Familiarize yourself with the files

**Q1:** Begin by reading in and parsing the relevant datasets and familiarizing yourself with them.  In this file, we will work with:

* `countries.xml`
* `topnames.xml`

You should name the variables representing the root nodes `countries_root` and `topnames_root`, respectively.

In [1]:
from lxml import etree
import os.path

datadir = "publicdata"

countries_path = os.path.join(datadir, "countries.xml")
parser = etree.XMLParser(remove_blank_text=True)
tree = etree.parse(countries_path, parser)

countries_root = tree.getroot()

topnames_path = os.path.join(datadir, "topnames.xml")
tree = etree.parse(topnames_path, parser)

topnames_root = tree.getroot()

In [2]:
# Testing cell
assert type(countries_root) is etree._Element
assert len(countries_root) == 231

assert type(topnames_root) is etree._Element
assert len(topnames_root) == 139

---

## Part C: `countries.xml`

**Q2:** Generate a list of all the country names in the `countries.xml` file, assigning to a variable `countries`.  Then, assign the number of countries to the variable `countrycount`.

In [3]:
countries = countries_root.xpath("//country/@name")
countrycount = len(countries)

In [4]:
# Testing cell
assert(countrycount == 231)
assert('Uruguay' in countries)

**Q3:** Write a function `findPop(root,countryName)` that finds the population of a given `country` in the dataset `countries.xml`. Use an XPath expression and a format string. Return your answer as an integer.

In [5]:
def findPop(root, countryName):
    pop = root.xpath(f"//country[@name='{countryName}']/@population")
    return int(pop[0])

In [6]:
# Testing cell
assert findPop(countries_root,'Cuba') == 10951334
assert findPop(countries_root,'Uruguay') == 3238952

**Q4:** Study the `countries` data carefully.  Then, use the `position()` function to create a node set consisting of, for countries in positions 5-55 inclusive, the population of the second city listed, if there are at least two cities listed.  (Note that you can use `and` inside the filter in `[]` for a given node.)

For example, nothing is in the node set for Aruba (no cities listed) or Armenia (only Yerevan listed), but Cordoba (a city in Argentina) is in the node set because Argentina has four cities listed.

Your answer should use a single XPath expression.  Store the results in a list `secondPops` of integers.

In [7]:
secondPops = countries_root.xpath("//country[position()>5 and position()<55 and count(city)>1]/city[2]/population/text()")
secondPops = [int(x) for x in secondPops]

# Print the list
print(secondPops)

[1208713, 1302000, 1599000, 2209465, 1000000, 1064255]


In [8]:
assert len(secondPops) == 6
assert secondPops[0] == 1208713
assert secondPops[5] == 1064255

> You've reached the first checkpoint in the lab.  Make sure to have it signed off by the instructor or TA.
>
> Checkpoint 1: How could you change the previous question's answer to instead return the names of the second cities, but only if the population is at least 1500000?

In [30]:
thirdPops = countries_root.xpath("//country[position()>5 and position()<55 and count(city)>1]/city[2]/population[text()>1500000]/../name/text()")

# Print the list
print(thirdPops)

['Chittagong', 'Salvador']


---

## Part D: `topnames.xml`

**Q5:** With reference to the `topnames` dataset, find all years where there was a count (either sex) that was strictly larger than 50,000.  Store the resulting list of years (as strings) in a variable `yearsList1`.

In [23]:
yearsList1 = topnames_root.xpath("//year/*/count[text()>50000]/../../@value")

# Print the beginning of the list
print(yearsList1)

['1915', '1916', '1917', '1918', '1919', '1920', '1921', '1922', '1923', '1924', '1925', '1926', '1927', '1928', '1929', '1930', '1931', '1932', '1933', '1934', '1935', '1936', '1937', '1938', '1939', '1940', '1941', '1942', '1943', '1944', '1945', '1946', '1947', '1948', '1949', '1950', '1951', '1952', '1953', '1954', '1955', '1956', '1957', '1958', '1959', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992']


In [24]:
# Testing cell
assert yearsList1[0] == '1915'
assert len(yearsList1) == 78

**Q6:** With reference to the `topnames` dataset, find all years where the top female name had a count that was strictly larger than 50,000. Store the resulting list of years (as strings) in a variable `yearsList2`.

In [25]:
yearsList2 = topnames_root.xpath("//year/sex[@value='Female']/count[text()>50000]/../../@value")

# Print the beginning of the list
print(yearsList2)

['1915', '1916', '1917', '1918', '1919', '1920', '1921', '1922', '1923', '1924', '1925', '1926', '1927', '1928', '1929', '1930', '1931', '1932', '1933', '1934', '1935', '1936', '1937', '1938', '1939', '1940', '1941', '1942', '1943', '1944', '1945', '1946', '1947', '1948', '1949', '1950', '1951', '1952', '1953', '1954', '1955', '1956', '1957', '1958', '1959', '1960', '1963', '1964', '1965', '1966', '1967', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1986', '1987', '1988']


In [26]:
# Testing cell
assert yearsList2[0] == '1915'
assert len(yearsList2) == 68

> You've reached the second (and final) checkpoint in the lab.  Make sure to have it signed off by the instructor or TA.
>
> Checkpoint 2: How could you instead find all years for which the combined male and female count was strictly larger than 50,000?

In [27]:
yearsList3 = topnames_root.xpath("//year[sum(sex/count) > 50000]/@value")
print(yearsList3)

['1912', '1913', '1914', '1915', '1916', '1917', '1918', '1919', '1920', '1921', '1922', '1923', '1924', '1925', '1926', '1927', '1928', '1929', '1930', '1931', '1932', '1933', '1934', '1935', '1936', '1937', '1938', '1939', '1940', '1941', '1942', '1943', '1944', '1945', '1946', '1947', '1948', '1949', '1950', '1951', '1952', '1953', '1954', '1955', '1956', '1957', '1958', '1959', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004']


---

---
## Part E

How much time (in minutes/hours) did you spend on this lab outside of class?

I finished this lab in class.